# Проверка сгенерированных данных
Пять графиков: сходятся ли цели, связан ли отказ с качеством заёмщика, реалистичны ли масштабы и связи между показателями.

**Оговорка:** данные синтетические, связи заданы правилами генератора. Графики показывают, что правила реализованы правильно и не противоречат здравому смыслу. Реальность банка они не доказывают.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

BLUE, ORANGE, GRAY, INK = "#2a6fdb", "#e8743b", "#b8bcc4", "#222831"
SEG = {"IE": "#8aa9d6", "LLC micro": "#2a6fdb", "LLC small": "#1b4a9c", "LLC medium": "#0e2a5c"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": "#e6e8ec", "grid.linewidth": 0.8, "axes.axisbelow": True})

d = pd.read_csv("data/applications.csv", dtype={"inn": str})
c = d.drop_duplicates("inn").copy()          # одна строка = одна компания
print(f"Заявок: {len(d):,}, компаний: {len(c):,}, заявок на компанию: {len(d)/len(c):.2f}")
print(f"Общая доля одобрения: {d.approved.mean():.1%}")

## 1. Доля одобрения по продуктам: цель и факт
Цели заданы для **первых попыток** обращения: калибровка свободного члена должна дать факт в пределах ±2 п.п. Повторные заявки после отказа одобряют реже, поэтому доля по всем заявкам ниже (серые точки). У FACTORING цель снижена, так как много жёстких отказов (см. README).

In [ ]:
import json
meta = json.load(open("data/dataset_meta.json"))
fact = pd.Series(meta["approval_rate_first_attempt_by_product"])       # 1-я попытка обращения
fact_all = d.groupby("product_type").approved.mean()                     # все заявки
spec = pd.Series(meta["approval_target_spec"]); eff = pd.Series(meta["approval_target_effective"])
order = fact.sort_values().index
n = d.product_type.value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(order))
ax.barh(y, fact[order], color=BLUE, height=0.55, label="Факт, 1-я попытка")
ax.scatter(fact_all[order], y, color=GRAY, s=30, label="Факт, все заявки", zorder=3)
ax.scatter(spec[order], y, marker="|", s=350, color=ORANGE, linewidths=2.5, label="Цель из спецификации", zorder=3)
for yi, p in zip(y, order):
    ax.text(fact[p] + 0.012, yi, f"{fact[p]:.0%}  (n={n[p]:,})", va="center", fontsize=9, color=INK)
ax.set_yticks(y, order); ax.set_xlim(0, 0.95); ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_title("Одобрение по продуктам: факт против цели"); ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3)
plt.tight_layout(); plt.show()

## 2. Чем хуже кредитная дисциплина, тем ниже одобрение
Главная проверка «качества»: просрочки в БКИ, налоговый долг, отрицательный капитал, подозрительные флаги. Ожидаем монотонное падение.

In [ ]:
dpd = pd.cut(d.bki_max_dpd_12m.fillna(-1), [-2, -.5, 0, 29, 89, 400],
             labels=["нет БКИ", "без просрочек", "1–29 дн.", "30–89 дн.", "90+ дн."])
risk = d.registry_risk_flag > 0
panels = [
  ("Просрочка в БКИ (макс. за 12 мес.)", d.approved.groupby(dpd, observed=True)),
  ("Налоговый долг", d.approved.groupby(d.tax_arrears_amount.gt(0).map({False: "нет", True: "есть"}))),
  ("Капитал (ООО с отч.)", d.approved[d.equity_ratio.notna()].groupby(
        d.equity_ratio[d.equity_ratio.notna()].lt(0).map({False: "≥ 0", True: "< 0"}))),
  ("Флаги риска (ООО)", d.approved[d.legal_form == "LLC"].groupby(
        risk[d.legal_form == "LLC"].map({False: "нет", True: "есть"}))),
]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=True, gridspec_kw={"width_ratios": [5, 2, 2, 2]})
for ax, (title, g) in zip(axes, panels):
    m = g.mean(); cols = [BLUE] * len(m)
    ax.bar(range(len(m)), m.values, color=cols, width=0.6)
    for i, v in enumerate(m.values): ax.text(i, v + 0.015, f"{v:.0%}", ha="center", fontsize=9)
    ax.set_xticks(range(len(m)), m.index, fontsize=8.5); ax.set_title(title, fontsize=9.5, wrap=True)
    ax.yaxis.set_major_formatter(PercentFormatter(1)); ax.set_ylim(0, 0.85); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Доля одобрения"); plt.tight_layout(); plt.show()

## 3. Возраст и рентабельность: молодые и убыточные получают отказы чаще
Жёсткие отказы (возраст < 3 мес.) дают нулевую долю, дальше одобрение плавно растёт с возрастом и прибыльностью.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
ab = pd.cut(d.company_age_months, [0, 2, 5, 11, 23, 47, 95, 191, 360],
            labels=["1–2", "3–5", "6–11", "12–23", "24–47", "48–95", "96–191", "192+"])
r = d.approved.groupby(ab, observed=True).mean()
a.plot(r.index.astype(str), r.values, marker="o", color=BLUE, lw=2, ms=7)
for x, v in zip(r.index.astype(str), r.values): a.annotate(f"{v:.0%}", (x, v), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
a.set_title("Возраст компании, мес."); a.set_ylabel("Доля одобрения"); a.yaxis.set_major_formatter(PercentFormatter(1)); a.set_ylim(0, 0.85)

L = d[d.net_margin.notna()]
mb = pd.cut(L.net_margin, [-.51, -.2, -.1, -.05, 0, .05, .1, .15, .2, .51])
r = L.approved.groupby(mb, observed=True).agg(["mean", "size"])
x = [f"{i.left:+.2f}…{i.right:+.2f}" for i in r.index]
b.bar(range(len(r)), r["mean"], color=BLUE, width=0.6)
for i, v in enumerate(r["mean"]): b.text(i, v + 0.015, f"{v:.0%}", ha="center", fontsize=9)
b.set_xticks(range(len(r)), x, rotation=35, ha="right", fontsize=8.5); b.set_ylim(0, 0.85)
b.set_title("Чистая рентабельность (ООО с отчётностью)"); b.yaxis.set_major_formatter(PercentFormatter(1)); b.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 4. Масштаб бизнеса: выручка по сегментам
Медианы должны быть около 3 млн (ИП), 9 млн (ООО micro), 250 млн (small), 1.1 млрд (medium). Шкала логарифмическая: выручка распределена логнормально.

In [ ]:
seg = np.select([c.legal_form.eq("IE"), c.msp_category.eq("micro"), c.msp_category.eq("small")],
                ["IE", "LLC micro", "LLC small"], "LLC medium")
bins = np.logspace(np.log10(c.revenue_last_year.min()), np.log10(c.revenue_last_year.max()), 70)
fig, ax = plt.subplots(figsize=(9.5, 4.3))
for name in SEG:
    v = c.revenue_last_year[seg == name]
    ax.hist(v, bins=bins, color=SEG[name], alpha=0.85, label=f"{name}: медиана {v.median()/1e6:,.1f} млн, n={len(v):,}")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Выручка за год, ₽ (лог. шкала)"); ax.set_ylabel("Компаний (лог. шкала)")
ax.set_title("Распределение выручки по сегментам"); ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

## 5. Связи между признаками: корреляции Spearman
Ожидаем: выручка ↔ налоги; долг в балансе ↔ долг в БКИ; рентабельность ↔ капитал; возраст ↔ выручка слабая. Остальные пары близки к нулю: признаки не должны быть случайным шумом, но и не должны дублировать друг друга.

In [ ]:
feat = pd.DataFrame({
    "ln выручка": np.log(c.revenue_last_year),     "ln налоги": np.log1p(c.taxes_paid_last_year),
    "ln возраст": np.log(c.company_age_months), "рост выручки": c.revenue_growth_yoy,
    "налоги / выручка": c.taxes_paid_last_year / c.revenue_last_year, "рентабельность": c.net_margin,
    "капитал / активы": c.equity_ratio, "тек. ликвидность": c.current_ratio, "долг / выручка": c.debt_to_revenue,
    "ln налог. долг": np.log1p(c.tax_arrears_amount), "ln долг ФССП": np.log1p(c.fssp_open_amount),
    "ln иски": np.log1p(c.arbitr_defendant_amount_12m),
    "ln долг БКИ": np.log1p(c.bki_total_debt), "макс. просрочка": c.bki_max_dpd_12m})
cm = feat.corr(method="spearman")                       # попарно, пропуски пропускаются
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm)), cm.columns, rotation=45, ha="right", fontsize=9); ax.set_yticks(range(len(cm)), cm.columns, fontsize=9)
for i in range(len(cm)):
    for j in range(len(cm)):
        v = cm.iat[i, j]; ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5, color="white" if abs(v) > 0.6 else INK)
ax.grid(False); ax.set_title("Spearman, одна строка = компания"); fig.colorbar(im, shrink=0.8)
plt.tight_layout(); plt.show()